<a href="https://colab.research.google.com/github/Shiv12GitHub39G/Bharat-Intern-Task1/blob/main/Self-Pruning%20Neural%20Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Self-Pruning Neural Network
============================
Author  : Shivam Gupta
Dataset : CIFAR-10
Framework: PyTorch
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time


# Reproducibility

torch.manual_seed(42)
np.random.seed(42)


# Device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# PART 1 — Prunable Layer


class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))

        # Better initialization (encourages pruning)
        self.gate_scores = nn.Parameter(
            torch.empty(out_features, in_features).normal_(mean=-1.0, std=0.1)
        )

        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

    def forward(self, x):
        # Temperature scaling (critical)
        gates = torch.sigmoid(5 * self.gate_scores)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self):
        return torch.sigmoid(5 * self.gate_scores).detach()

    def sparsity_level(self, threshold=0.05):
        g = self.get_gates()
        return (g < threshold).float().mean().item()


# Model


class SelfPruningNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = PrunableLinear(3072, 1024)
        self.bn1 = nn.BatchNorm1d(1024)

        self.fc2 = PrunableLinear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)

        self.fc3 = PrunableLinear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)

        self.fc4 = PrunableLinear(256, 10)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.view(x.size(0), -1)

        x = self.dropout(F.relu(self.bn1(self.fc1(x))))
        x = self.dropout(F.relu(self.bn2(self.fc2(x))))
        x = self.dropout(F.relu(self.bn3(self.fc3(x))))
        x = self.fc4(x)

        return x

    def prunable_layers(self):
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                yield m

    def sparsity_loss(self):
        total = 0
        count = 0
        for layer in self.prunable_layers():
            gates = torch.sigmoid(5 * layer.gate_scores)
            total += gates.sum()
            count += gates.numel()
        return total / count  # normalized

    def overall_sparsity(self, threshold=0.05):
        total, pruned = 0, 0
        for layer in self.prunable_layers():
            g = layer.get_gates()
            pruned += (g < threshold).sum().item()
            total  += g.numel()
        return pruned / total

# Data


def get_loaders(batch_size=128):
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    test  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

    return (
        DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=2),
        DataLoader(test, batch_size=256, shuffle=False, num_workers=2)
    )


# Train / Eval


def train_epoch(model, loader, opt, lam):
    model.train()

    total_loss, correct, total = 0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        opt.zero_grad()

        out = model(x)

        ce = F.cross_entropy(out, y)
        sp = model.sparsity_loss()

        loss = ce + lam * sp
        loss.backward()
        opt.step()

        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)

        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)

    return total


# Experiment


def run_experiment(lam, train_loader, test_loader, epochs=30):
    print(f"\nTraining with λ = {lam}")

    model = SelfPruningNet().to(device)
    opt = optim.Adam(model.parameters(), lr=5e-4)

    for epoch in range(epochs):
        loss, acc = train_epoch(model, train_loader, opt, lam)

        if epoch % 5 == 0:
            sp = model.overall_sparsity()
            print(f"Epoch {epoch} | Loss {loss:.4f} | Acc {acc*100:.2f}% | Sparsity {sp*100:.2f}%")

    test_acc = evaluate(model, test_loader)
    sparsity = model.overall_sparsity()

    return {
        "lambda": lam,
        "acc": test_acc,
        "sparsity": sparsity,
        "model": model
    }



if __name__ == "__main__":

    train_loader, test_loader = get_loaders()

    # lambda values
    LAMBDAS = [1e-8, 5e-8, 1e-7]

    results = []

    for lam in LAMBDAS:
        res = run_experiment(lam, train_loader, test_loader)
        results.append(res)

    print("\nRESULTS")
    print("="*50)
    for r in results:
        print(f"λ={r['lambda']} | Acc={r['acc']*100:.2f}% | Sparsity={r['sparsity']*100:.2f}%")

Using device: cpu


100%|██████████| 170M/170M [00:02<00:00, 71.2MB/s]



Training with λ = 1e-08
Epoch 0 | Loss 2.1513 | Acc 27.61% | Sparsity 99.99%
Epoch 5 | Loss 1.5129 | Acc 45.43% | Sparsity 99.98%
Epoch 10 | Loss 1.4257 | Acc 48.61% | Sparsity 99.97%
Epoch 15 | Loss 1.3780 | Acc 50.29% | Sparsity 99.97%
Epoch 20 | Loss 1.3484 | Acc 51.61% | Sparsity 99.97%
Epoch 25 | Loss 1.3095 | Acc 52.98% | Sparsity 99.96%

Training with λ = 5e-08
Epoch 0 | Loss 2.1512 | Acc 28.12% | Sparsity 100.00%
Epoch 5 | Loss 1.5133 | Acc 45.37% | Sparsity 99.98%
Epoch 10 | Loss 1.4316 | Acc 48.39% | Sparsity 99.97%
Epoch 15 | Loss 1.3802 | Acc 50.27% | Sparsity 99.97%
Epoch 20 | Loss 1.3396 | Acc 51.82% | Sparsity 99.96%
Epoch 25 | Loss 1.3079 | Acc 53.07% | Sparsity 99.96%

Training with λ = 1e-07
Epoch 0 | Loss 2.1497 | Acc 27.79% | Sparsity 99.99%
Epoch 5 | Loss 1.5117 | Acc 45.35% | Sparsity 99.98%
Epoch 10 | Loss 1.4280 | Acc 48.26% | Sparsity 99.97%
Epoch 15 | Loss 1.3819 | Acc 50.19% | Sparsity 99.97%
Epoch 20 | Loss 1.3380 | Acc 51.71% | Sparsity 99.96%
Epoch 25 | L

In [2]:
import numpy as np
import matplotlib.pyplot as plt

COLORS = ["#378ADD", "#EF9F27", "#E24B4A"]

# -------------------------------
# Plot 1 — Gate Distributions
# -------------------------------
def plot_gate_distributions(results):
    fig, axes = plt.subplots(1, len(results), figsize=(15, 4))

    for ax, r, c in zip(axes, results, COLORS):
        all_gates = np.concatenate([
            layer.get_gates().cpu().numpy().flatten()
            for layer in r["model"].prunable_layers()
        ])

        ax.hist(all_gates, bins=100, color=c, alpha=0.8)
        ax.axvline(x=0.05, linestyle="--", color="black")

        ax.set_title(f"λ={r['lambda']}\nS={r['sparsity']*100:.1f}% | A={r['test_acc']*100:.1f}%")
        ax.set_xlabel("Gate value")
        ax.set_ylabel("Count")

    plt.tight_layout()
    plt.savefig("gate_distribution.png")
    plt.show()


# -------------------------------
# Plot 2 — Accuracy vs Sparsity
# -------------------------------
def plot_tradeoff(results):
    x = [r["sparsity"] * 100 for r in results]
    y = [r["test_acc"] * 100 for r in results]

    plt.figure(figsize=(7,5))

    for r, c in zip(results, COLORS):
        plt.scatter(r["sparsity"]*100, r["test_acc"]*100, s=120, color=c)
        plt.text(r["sparsity"]*100, r["test_acc"]*100,
                 f"λ={r['lambda']}", fontsize=9)

    plt.xlabel("Sparsity (%)")
    plt.ylabel("Accuracy (%)")
    plt.title("Accuracy vs Sparsity Tradeoff")

    plt.savefig("tradeoff.png")
    plt.show()


# -------------------------------
# Plot 3 — Bar Chart
# -------------------------------
def plot_bar(results):
    labels = [f"λ={r['lambda']}" for r in results]
    acc = [r["test_acc"]*100 for r in results]
    sp  = [r["sparsity"]*100 for r in results]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax1 = plt.subplots()

    ax2 = ax1.twinx()

    ax1.bar(x - width/2, acc, width, label="Accuracy")
    ax2.bar(x + width/2, sp, width, label="Sparsity")

    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)

    ax1.set_ylabel("Accuracy (%)")
    ax2.set_ylabel("Sparsity (%)")

    plt.title("Lambda Comparison")

    plt.savefig("bar_chart.png")
    plt.show()


# -------------------------------
# Plot 4 — Per-layer Sparsity
# -------------------------------
def plot_layer_sparsity(results):
    data = []

    for r in results:
        layer_vals = []
        for layer in r["model"].prunable_layers():
            layer_vals.append(layer.sparsity_level() * 100)
        data.append(layer_vals)

    data = np.array(data)

    plt.imshow(data, cmap="viridis", aspect="auto")

    plt.xticks(range(data.shape[1]), ["fc1","fc2","fc3","fc4"])
    plt.yticks(range(len(results)), [f"λ={r['lambda']}" for r in results])

    plt.colorbar(label="Sparsity (%)")
    plt.title("Layer-wise Sparsity")

    plt.savefig("layer_sparsity.png")
    plt.show()


# -------------------------------
# Plot 5 — Best Model Gates
# -------------------------------
def plot_best_model(results):
    best = max(results, key=lambda r: r["test_acc"])

    all_gates = np.concatenate([
        layer.get_gates().cpu().numpy().flatten()
        for layer in best["model"].prunable_layers()
    ])

    plt.figure(figsize=(7,5))
    plt.hist(all_gates, bins=120)

    plt.axvline(x=0.05, linestyle="--")

    plt.title(f"Best Model (λ={best['lambda']})")
    plt.xlabel("Gate value")
    plt.ylabel("Count")

    plt.savefig("best_model.png")
    plt.show()


# -------------------------------
# RUN ALL PLOTS
# -------------------------------
def run_all_plots(results):
    plot_gate_distributions(results)
    plot_tradeoff(results)
    plot_bar(results)
    plot_layer_sparsity(results)
    plot_best_model(results)

